![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 3 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">E3.4 · Detección de anomalías</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Objetivo** | Detectar anomalías con reglas estadísticas, Isolation Forest y LOF, evaluarlas con etiquetas cuando existen y fijar un umbral según la capacidad de revisión del equipo. |
| **Resultado de aprendizaje** | RDA2 · competencias CG-G2 y CE-G2 (según el sílabo) |
| **Duración** | ≈ 4 h |
| **Teoría** | Manual M3 §8–9 · Animaciones A3.7 y A3.8 · Video V3.3 |
| **Datos** | Sintéticos · **KDD Cup 99**, subconjunto `SA` (`fetch_kddcup99`: 100 655 conexiones de red, 3.4 % ataques) |

**Niveles:** 1 · z-score, IQR e Isolation Forest de juguete desde cero → 2 · `IsolationForest` y `LocalOutlierFactor` → 3 · intrusiones de red: PR-AUC y falsas alarmas → 4 · reto: umbral según las alertas que el equipo puede revisar.

## 0 · Configuración

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## Nivel 1 · Desde cero (prelaboratorio)

### 1.1 Reglas estadísticas
- **z-score:** marcar si $|x - \bar x| / s > 3$. Supone una distribución aproximadamente normal, y los propios atípicos inflan $\bar x$ y $s$.
- **IQR (Tukey):** marcar si $x < Q_1 - 1.5\,\text{IQR}$ o $x > Q_3 + 1.5\,\text{IQR}$. Usa cuantiles, así que es más robusta.

In [ ]:
rng = np.random.default_rng(SEED)
tiempos = np.concatenate([rng.lognormal(mean=3.0, sigma=0.5, size=950), rng.uniform(70, 300, size=50)])   # ms de respuesta; 50 lentitudes reales
es_anomalia = np.r_[np.zeros(950, bool), np.ones(50, bool)]


def z_score(x, umbral=3.0):
    # TODO: devuelve una máscara booleana con |x − media| / desviación > umbral.
    return ...


def iqr(x, k=1.5):
    # TODO: calcula Q1, Q3 e IQR con np.quantile y marca lo que cae fuera de [Q1 − k·IQR, Q3 + k·IQR].
    q1, q3 = ...
    rango = ...
    return ...


for nombre, marca in (("z-score > 3", z_score(tiempos)), ("IQR 1.5", iqr(tiempos)), ("IQR sobre log(x)", iqr(np.log(tiempos)))):
    vp = np.sum(marca & es_anomalia)
    print(f"{nombre:18s} marca {marca.sum():3d} · detecta {vp}/50 anomalías · falsas alarmas {marca.sum() - vp}")

**Qué observar.** El z-score se pierde un tercio de las anomalías: ellas mismas inflan la desviación estándar y "se esconden" (efecto de **enmascaramiento**). El IQR sobre los datos crudos las atrapa todas, pero como la variable es asimétrica marca también decenas de tiempos normales de la cola. Aplicado sobre $\log(x)$ equilibra ambos errores. Y ninguna regla univariada ve anomalías que solo aparecen en la **combinación** de variables.

### 1.2 Isolation Forest de juguete
Idea: una anomalía es **fácil de aislar**. Un árbol de aislamiento elige una variable al azar y un corte al azar entre su mínimo y su máximo, y repite hasta que cada punto queda solo. La **longitud del camino** $h(x)$ es cuántos cortes hicieron falta. Promediando sobre muchos árboles, el puntaje es

$$ s(x) = 2^{-\,\overline{h}(x)/c(\psi)}, \qquad c(\psi) = 2H(\psi - 1) - \frac{2(\psi - 1)}{\psi}, $$

donde $\psi$ es el tamaño de la submuestra y $H$ el número armónico. Puntajes cercanos a 1 indican anomalía.

In [ ]:
def c_psi(n):
    return 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n if n > 2 else (1.0 if n == 2 else 0.0)


def camino(x, X, r, prof=0, limite=12):
    """Longitud del camino de x en un árbol de aislamiento construido al vuelo sobre X."""
    # TODO: si queda ≤ 1 punto o se llega al límite, devuelve prof + c_psi(len(X)); si no, corta al azar y sigue por el lado de x.
    if len(X) <= 1 or prof >= limite:
        return ...
    j = ...
    ...


def puntaje_iforest(x, X, arboles=100, psi=256, semilla=SEED):
    r = np.random.default_rng(semilla)
    h = [camino(x, X[r.choice(len(X), min(psi, len(X)), replace=False)], r) for _ in range(arboles)]
    return 2 ** (-np.mean(h) / c_psi(min(psi, len(X)))), float(np.mean(h))


from sklearn.datasets import make_blobs

X2, _ = make_blobs(n_samples=500, centers=[[0, 0], [5, 5]], cluster_std=1.0, random_state=SEED)
atipicos = rng.uniform(-6, 11, size=(15, 2))
atipicos = atipicos[np.min(np.linalg.norm(atipicos[:, None] - np.array([[0, 0], [5, 5]]), axis=2), axis=1) > 3.5]
X2 = np.vstack([X2, atipicos])
y2 = np.r_[np.zeros(500, int), np.ones(len(atipicos), int)]
normal, raro = X2[0], X2[-1]
for nombre, x in (("punto normal", normal), ("anomalía", raro)):
    s, h = puntaje_iforest(x, X2)
    print(f"{nombre}: camino medio {h:.1f} cortes · puntaje {s:.3f}")

## Nivel 2 · Con scikit-learn

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import LocalOutlierFactor
from scipy.stats import spearmanr

iso = IsolationForest(n_estimators=200, random_state=SEED).fit(X2)
s_sk = -iso.score_samples(X2)                   # scikit-learn devuelve el negativo del puntaje: lo invertimos
s_prop = np.array([puntaje_iforest(x, X2, arboles=60)[0] for x in X2])
rho = spearmanr(s_sk, s_prop).statistic
print(f"correlación de rangos entre tu Isolation Forest y el de scikit-learn: {rho:.3f}")
assert rho > 0.85, "Tu puntaje debería ordenar los puntos de forma parecida al de scikit-learn."

lof = LocalOutlierFactor(n_neighbors=20)
lof.fit_predict(X2)
s_lof = -lof.negative_outlier_factor_
tabla_2d = pd.DataFrame({"Isolation Forest": {"ROC-AUC": roc_auc_score(y2, s_sk), "PR-AUC": average_precision_score(y2, s_sk)},
                         "LOF (k = 20)": {"ROC-AUC": roc_auc_score(y2, s_lof), "PR-AUC": average_precision_score(y2, s_lof)}}).T.round(3)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
for ax, (nombre, s) in zip(axes, (("Isolation Forest", s_sk), ("LOF", s_lof))):
    sc = ax.scatter(X2[:, 0], X2[:, 1], c=s, cmap="RdPu", s=12)
    ax.scatter(X2[y2 == 1, 0], X2[y2 == 1, 1], s=90, facecolors="none", edgecolors=UEES["azul"], label="anomalía real")
    ax.set(title=f"{nombre}: puntaje de anomalía")
    ax.legend(loc="lower right")
    fig.colorbar(sc, ax=ax)
plt.tight_layout()
plt.show()
tabla_2d

**Qué observar.** Tu árbol de juguete ordena los puntos casi igual que `IsolationForest`. El parámetro `contamination` solo fija el **umbral** para convertir puntajes en etiquetas (qué fracción marcar); no cambia el orden. Para evaluar, trabaja con los puntajes (`score_samples`, `negative_outlier_factor_`).

## Nivel 3 · Datos reales: intrusiones de red (KDD Cup 99, subconjunto SA)

Cada fila es una conexión de red descrita por 41 variables (duración, protocolo, bytes enviados, errores, conexiones al mismo servicio en los últimos segundos…). El subconjunto `SA` es mayoritariamente tráfico normal con 3.4 % de ataques. Entrenamos **sin etiquetas** y usamos las etiquetas solo para evaluar.

In [ ]:
from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split

kdd = fetch_kddcup99(subset="SA", as_frame=True, percent10=True, random_state=SEED)
K = kdd.frame.copy()
texto = lambda v: v.decode() if isinstance(v, bytes) else str(v)        # el dataset trae textos como bytes (b'tcp')
categoricas = ["protocol_type", "service", "flag"]
for c in categoricas:
    K[c] = K[c].map(texto)
K["tipo"] = K["labels"].map(texto).str.rstrip(".")
numericas = [c for c in K.columns if c not in categoricas + ["labels", "tipo"]]
K[numericas] = K[numericas].apply(pd.to_numeric)
K["ataque"] = (K["tipo"] != "normal").astype(int)
print(f"{len(K)} conexiones · {K['ataque'].mean():.1%} ataques · tipos de ataque más frecuentes:")
print(K.loc[K.ataque == 1, "tipo"].value_counts().head(5).to_dict())

In [ ]:
# variables numéricas con log1p en los conteos de bytes (colas muy largas)
Xk = K[numericas].copy()
for c in ("src_bytes", "dst_bytes", "duration"):
    Xk[c] = np.log1p(Xk[c])
Xk_tr, Xk_te, yk_tr, yk_te = train_test_split(Xk, K["ataque"], test_size=0.3, stratify=K["ataque"], random_state=SEED)
tipos_te = K.loc[Xk_te.index, "tipo"].to_numpy()

from sklearn.preprocessing import StandardScaler


def detectores(X_entrenamiento, semilla=SEED):
    """Ajusta los tres detectores con X_entrenamiento y devuelve sus puntajes sobre la prueba (mayor = más anómalo)."""
    esc = StandardScaler().fit(X_entrenamiento)
    iso_ = IsolationForest(n_estimators=200, random_state=semilla).fit(X_entrenamiento)
    sub = np.random.default_rng(semilla).choice(len(X_entrenamiento), min(20000, len(X_entrenamiento)), replace=False)
    lof_ = LocalOutlierFactor(n_neighbors=35, novelty=True).fit(esc.transform(X_entrenamiento.iloc[sub]))   # LOF es costoso: submuestra
    Zte = esc.transform(Xk_te)
    return {"máximo |z| (baseline)": np.abs(Zte).max(axis=1), "Isolation Forest": -iso_.score_samples(Xk_te), "LOF (k = 35)": -lof_.score_samples(Zte)}


def evaluar(puntajes):
    """ROC-AUC, PR-AUC y fracción detectada de cada tipo de ataque con un umbral que admite 1 % de falsas alarmas."""
    y = yk_te.to_numpy()
    filas = {}
    for nombre, sc in puntajes.items():
        umbral = np.quantile(sc[y == 0], 0.99)
        filas[nombre] = {"ROC-AUC": roc_auc_score(y, sc), "PR-AUC": average_precision_score(y, sc),
                         **{f"detecta {t}": np.mean(sc[tipos_te == t] > umbral) for t in ("smurf", "neptune", "satan")}}
    return pd.DataFrame(filas).T.round(3)


# Escenario A · detección de atípicos: se entrena con TODO el tráfico, sin saber qué es ataque
puntajes_a = detectores(Xk_tr)
tabla_a = evaluar(puntajes_a)
print(f"prevalencia de ataques en prueba: {yk_te.mean():.3f} (PR-AUC del azar)")
tabla_a

**Qué observar.** Con el entrenamiento contaminado, los tres métodos **no detectan `smurf`**, que es el 73 % de los ataques, y LOF queda por debajo del azar. La razón está en los datos: `smurf` son miles de conexiones ICMP casi idénticas (96 % duplicadas, `count` = 511). Para el detector no son raras: forman una región **densa** que termina formando parte de lo "normal" (**enmascaramiento**). Además inflan la media y la desviación con las que se escalan las variables.

### 3.1 Detección de novedades: aprender solo de lo normal
Si se dispone de un período de tráfico **verificado como normal** (por ejemplo, una semana auditada), se puede entrenar solo con él y marcar lo que se aparte. Es la **detección de novedades** (en scikit-learn, `LocalOutlierFactor(novelty=True)` o cualquier detector entrenado con datos limpios).

In [ ]:
# Escenario B · detección de novedades: entrenamos solo con conexiones normales (aquí usamos las etiquetas para simular ese período limpio)
puntajes_b = detectores(Xk_tr[yk_tr.to_numpy() == 0])
tabla_b = evaluar(puntajes_b)
tabla_b

**Qué observar.** Con un entrenamiento limpio todo mejora, y el **baseline más simple** (el |z| más extremo de la conexión, con media y desviación del tráfico normal) es el mejor: detecta todos los `smurf` porque su `count` está muy lejos de lo normal. Isolation Forest mejora mucho en PR-AUC, pero sigue sin marcar `smurf` con 1 % de falsas alarmas. Dos lecciones: los métodos no supervisados detectan lo **raro**, no necesariamente lo **malicioso**, y siempre hay que compararlos con un baseline sencillo.

### 3.2 El costo de las falsas alarmas

In [ ]:
def alarmas_para_recall(y, s, recall_objetivo):
    """Umbral con el recall pedido y falsas alarmas por cada 10 000 conexiones normales."""
    orden = np.sort(s[y == 1])[::-1]
    umbral = orden[int(np.ceil(recall_objetivo * len(orden))) - 1]
    fp = np.sum((s >= umbral) & (y == 0))
    return umbral, fp / np.sum(y == 0) * 10000


filas = []
for rec in (0.5, 0.8, 0.9, 0.95):
    fila = {"recall pedido": rec}
    for nombre in ("máximo |z| (baseline)", "Isolation Forest"):
        _, fa = alarmas_para_recall(yk_te.to_numpy(), puntajes_b[nombre], rec)
        fila[f"falsas alarmas por 10 000 normales · {nombre}"] = round(fa)
    filas.append(fila)
tabla_fa = pd.DataFrame(filas).set_index("recall pedido")
tabla_fa

**Qué observar.** Para el mismo recall del 80 %, Isolation Forest genera ≈ 7 veces más falsas alarmas que el baseline (175 frente a 24 por cada 10 000 conexiones normales), y pasar del 90 % al 95 % de recall triplica las del baseline. En un centro de operaciones de seguridad (SOC) las falsas alarmas tienen un costo real: horas de analistas y **fatiga de alertas**, que hace que se ignoren las verdaderas.

## Nivel 4 · Reto: el umbral lo fija la capacidad del equipo

Supón que el tráfico se revisa en lotes de 10 000 conexiones y que el equipo de seguridad puede investigar como máximo **N alertas por lote**. El umbral ya no es 0.5 ni un `contamination` arbitrario: son los **N puntajes más altos**.

In [ ]:
def desempeno_por_capacidad(y, s, alertas_por_10k):
    n = int(round(alertas_por_10k * len(y) / 10000))
    top = np.argsort(-s)[:n]
    vp = int(np.sum(y[top]))
    return {"alertas por lote": alertas_por_10k, "recall": vp / y.sum(), "precisión": vp / n}


y_arr = yk_te.to_numpy()
capacidades = (50, 100, 200, 300, 400, 600)
mejor = "máximo |z| (baseline)"
tabla_cap = pd.DataFrame([desempeno_por_capacidad(y_arr, puntajes_b[mejor], c) for c in capacidades]).set_index("alertas por lote").round(3)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(tabla_cap.index, tabla_cap["recall"], "o-", color=UEES["vino"], label="recall")
ax.plot(tabla_cap.index, tabla_cap["precisión"], "o-", color=UEES["azul"], label="precisión")
ax.axvline(10000 * y_arr.mean(), ls="--", color=UEES["gris"], label=f"ataques reales por lote ≈ {10000 * y_arr.mean():.0f}")
ax.set(title="Detector |z| (novedades) según la capacidad de revisión", xlabel="alertas que el equipo revisa por cada 10 000 conexiones", ylim=(0, 1.05))
ax.legend()
plt.show()
tabla_cap

**Qué observar.** Con capacidad menor que el número de ataques reales (≈ 340 por lote), el recall está limitado aunque la precisión sea alta; por encima, cada alerta extra rinde menos. Esta curva es la que se lleva a la conversación con el equipo: *"con 300 alertas por lote detectamos este porcentaje de ataques; para llegar a tanto necesitamos más analistas o un mejor modelo"*.

## Autoverificación

In [ ]:
assert iqr(np.log(tiempos)).sum() < iqr(tiempos).sum(), "Transformar con logaritmo debería reducir las marcas del IQR."
assert rho > 0.85
assert tabla_a.loc["Isolation Forest", "detecta smurf"] < 0.1, "Con entrenamiento contaminado, smurf debería quedar enmascarado."
assert tabla_b.loc["Isolation Forest", "PR-AUC"] > tabla_a.loc["Isolation Forest", "PR-AUC"], "Entrenar solo con lo normal debería mejorar a Isolation Forest."
assert tabla_b["PR-AUC"].max() > 10 * yk_te.mean(), "El mejor detector de novedades debería superar ampliamente al azar."
assert tabla_cap["recall"].is_monotonic_increasing, "Más capacidad no puede bajar el recall."
print("✓ E3.4 completo")

## Lista de cotejo (autoevaluación)

- [ ] Comparaste z-score e IQR y viste el efecto de la asimetría.
- [ ] Tu Isolation Forest de juguete ordena como el de scikit-learn.
- [ ] Evaluaste Isolation Forest y LOF con PR-AUC frente a un baseline, con entrenamiento contaminado y limpio.
- [ ] Analizaste por tipo de ataque qué detecta cada método y explicaste el enmascaramiento de `smurf`.
- [ ] Fijaste el umbral por capacidad de revisión y lo explicaste en términos del equipo.

**Reflexión:** en tu organización, ¿quién revisaría las alertas de un detector de anomalías y cuántas podría atender por día?

**Cómo te prepara para la Tarea 3:** la tarea exige comparar Isolation Forest y LOF, justificar el umbral y analizar a quién afectan las anomalías.

## Desafío opcional con IA agéntica · Novedades por error de reconstrucción

**Objetivo.** Probar un detector basado en reconstruir el tráfico normal y compararlo con los del notebook.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
En el notebook resuelto E3.4 (detección de anomalías con KDD Cup 99), agrega un detector de novedades por error de reconstrucción: un MLPRegressor que reconstruye las variables escaladas, entrenado solo con tráfico normal. Compara su PR-AUC y sus falsas alarmas por cada 10 000 conexiones normales, con un recall del 80 %, frente a Isolation Forest, LOF y el baseline del notebook. Explica en español si vale la pena su complejidad.
```

**Cómo verificar el resultado**

- El detector se entrena solo con tráfico normal.
- Todos los métodos se evalúan con las mismas particiones.
- La conclusión lo compara con el baseline simple, no solo con los otros detectores.

**Declara el uso de IA** (norma f del sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.